# What does this Script do?

## Imports the Relevant data from the Partners in care Looker database, cleans and combines it, then updates two SQL tables (daily_activity and referrals_activity) where its used to maintain a PowerBI Dashboard.  

import python packages

In [1]:
import pandas as pd
import os
import json
import datetime
from datetime import datetime, date, timezone
import numpy as np
import pytz


from io import StringIO
import looker_sdk

Custom packages

In [ ]:
from ds_general_utils import DsEmail
from ds_general_utils import ConfigTools
from ds_general_utils import Task, TaskManager
from ds_general_utils import MultiThreadedLogger


from utils.sql_connection import PicHorneDashesConnection

initialize task tracker

The task tracker allows for watching individual cell executions and seeing which ones fail, as well as preventing future cells from running if an error occurs. it also generates a summary email upon failure. 

In [3]:
# initialization of the task manager with the overall name of the task this code is supposed to do.
task_mgr = TaskManager("partners in care daily activity data ingestion")

initialize logger

This cell initializes the logger, which is used to log information, warnings, and errors during the execution of the script. It creates a log file with a timestamp in the filename and sets up the logger with a specific format and logging level. If the logging directory doesn't exist it creates it and moves all but the last 5 logs to the archive

In [4]:
task = Task(task_nm="Initialize log file", task_details="")
try:
    # creating any relevant dates
    now = datetime.now()
    now_str = now.strftime("%Y%m%d_%H%M%S")
    # log file info
    logging_dir = os.path.join(os.getcwd(), "logs")
    logging_filename = "partners_in_care_daily_activity_data_ingestion_{}.log".format(
        now_str
    )
    logging_filename_with_path = os.path.join(logging_dir, logging_filename)

    logger_manager = MultiThreadedLogger(
        logging_filename_with_path,
        "Partners_In_Care",
        "Daily_Activity_Data_ingestion",
        level="INFO",
        format_string="%(asctime)s %(levelname)s %(message)s",
    )
    logger = logger_manager.logger
    # updating the task tracker
    task["Details"] = "Successfully created and began writing to log file"
    task["Success"] = True
    logger.info("Data ingestion for partners in care daily activity started")

except Exception as e:
    task["Details"] = str(e)
    print(e)

task_mgr.add_task(task)

initialize variables

The below cell initializes starting global variables such as directory paths, configuration objects, and various date and time variables. It also updates the task tracker with the status of the initialization process.

In [5]:
task = Task(task_nm="Initialize starting global variables", task_details="")
try:
    task_mgr.check_errors()
    main_directory_path = str(os.getcwd())
    config_manager = ConfigTools(main_directory_path)
    config_object = config_manager.config_data

    looker_config_path = main_directory_path + "\\utils\\looker.ini"

    # variables used in pulling records from looker
    model = "oahu_connection_model"
    view = "base"
    limit = 100000

    sql_username = config_object["Josh"]["username"]
    sql_password = config_object["Josh"]["password"]

    today = date.today()
    today_str = today.strftime("%m_%d_%Y")
    reporting_period_today_str = today.strftime("%Y-%m-%d")

    now_utc = datetime.now(timezone.utc)
    hst = pytz.timezone("Pacific/Honolulu")
    now_hst = datetime.now(hst)
    now_utc_str = now_utc.strftime("%Y/%m/%d %H:%M:%S")
    now_str_formatted = now.strftime("%Y/%m/%d %H:%M:%S")
    now_hst_str = now_hst.strftime("%Y/%m/%d %H:%M:%S")

    # updating the task tracker
    task["Details"] = "successfully initialize global variables"
    task["Success"] = True
    logger.info("Initialized starting global variables")
except Exception as e:
    task["Details"] = str(e)
    logger.error(f"Initializing global variables failed with the following error :{e}")
    print(e)

task_mgr.add_task(task)

Grab program level information 

The below cell connects to the database and ingests client level data from Looker. It initializes the Looker SDK, defines the query to be executed, runs the query, processes the results, and merges the data into a single DataFrame. The task tracker is updated with the status of the data ingestion process.

In [ ]:
task = Task(
    task_nm="Connect to database and ingest program level data",
    task_details="",
)
try:
    task_mgr.check_errors()
    # initialize sdk
    sdk = looker_sdk.init40(looker_config_path)
    # Define fields
    program_fields = [
        "clients.id",
        "static_demographics.veteran_text",
        "enrollments.ref_household",
        "household_move_in_date.move_in_date",
        "household_makeup.household_type",
        "enrollments.reporting_period_start_date",
        "enrollments.reporting_period_end_date",
        "programs.project_type_code",
        "services.service_id",
        "services.start_date_date",
        "enrollments.start_date",
        "enrollments.end_date",
        "entry_screen.relationship_to_hoh",
        "clients.age",
    ]
    filter = {
        "enrollments.date_filter": "1 days",
        "programs.name": "-Emergency Housing Voucher CE",
    }

    # Function to generate query body
    def generate_body(model, view, program_fields, filter, limit):
        return {
            "model": model,
            "view": view,
            "fields": program_fields,
            "filters": filter,
            "limit": limit,
        }

    # Function to run the query
    def run_query(sdk, model, view, program_fields, filter, limit):
        body = generate_body(model, view, program_fields, filter, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results (add Reporting Period column)
    def process_result(result):

        df = pd.read_csv(StringIO(result))
        return df

    # Collect all DataFrames
    all_dataframes = []

    # Run query and process results for each date range
    result = run_query(sdk, model, view, program_fields, filter, limit)
    df = process_result(result)
    all_dataframes.append(df)

    # Concatenate all DataFrames into one
    client_level_df = pd.concat(all_dataframes)

    # updating the task tracker
    task["Details"] = "Successfully grabbed program data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled program data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"connecting to database and grabbing program data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

getting the referrals from the past 24 hours

The below cell connects to the database and ingests referral level data from Looker. It initializes the Looker SDK, defines the query to be executed, runs the query, processes the results, and merges the data into a single DataFrame. The task tracker is updated with the status of the data ingestion process.

In [ ]:
task = Task(
    task_nm="Connect to database and ingest referral level data",
    task_details="",
)
try:
    task_mgr.check_errors()
    # initialize sdk
    sdk = looker_sdk.init40(looker_config_path)
    # Define fields
    program_fields = [
        "client.count",
    ]
    filter = {
        "programs.name": "-Emergency Housing Voucher CE",
        "programs.project_type_code": "Coordinated Entry",
        "client_last_enrollment_by_type.is_latest_enrollment_by_type": "Yes",
        "referrals.Currently_On_Queue": "Yes",
        "enrollments.date_filter": "1 days",
    }
    view = "client"

    # Function to generate query body
    def generate_body(model, view, program_fields, filter, limit):
        return {
            "model": model,
            "view": view,
            "fields": program_fields,
            "filters": filter,
            "limit": limit,
        }

    # Function to run the query
    def run_query(sdk, model, view, program_fields, filter, limit):
        body = generate_body(model, view, program_fields, filter, limit)
        result = sdk.run_inline_query(result_format="csv", body=body)
        return result

    # Function to process the results (add Reporting Period column)
    def process_result(result):

        df = pd.read_csv(StringIO(result))
        return df

    # Collect all DataFrames
    all_dataframes = []

    # Run query and process results for each date range
    result = run_query(sdk, model, view, program_fields, filter, limit)
    df = process_result(result)
    all_dataframes.append(df)

    # Concatenate all DataFrames into one
    referrals_df = pd.concat(all_dataframes)

    # updating the task tracker
    task["Details"] = "Successfully grabbed referral data from looker"
    task["Success"] = True
    logger.info("Connected to database and pulled referral data from looker")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"Connecting to database and grabbing referral data failed with the following error :{e}"
    )
    print(e)

task_mgr.add_task(task)

This cell uploads the results stored in the `referrals_df` and `Client_level_df` DataFrames to the `partners_in_care_daily_activity` and `partners_in_care_referral_activity` tables in the database. It adds update timestamps in HST, CST, and UTC time zones, and then inserts or updates the data into the database using the `GsAutomationReliefAssistanceConnection` class. The task tracker is updated with the status of the database insertion process.

In [ ]:
task = Task(
    task_nm="uploading client table to database",
    task_details="",
)
try:
    task_mgr.check_errors()

    conn = PicHorneDashesConnection()

    client_level_df["Update Time HST"] = now_hst_str
    client_level_df["Update Time CST"] = now_str_formatted
    client_level_df["Update Time UTC"] = now_utc_str

    conn.insert_with_progress(
        df=client_level_df,
        schema="dbo",
        tbl_nm="partners_in_care_daily_activity",
        dtypes={},
    )

    # updating the task tracker
    task["Details"] = "Successfully updated client data in database"
    task["Success"] = True
    logger.info("Updated client data in Database")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"uploading new client data to database failed with the following error:{e}"
    )
    print(e)

task_mgr.add_task(task)

In [ ]:
task = Task(
    task_nm="uploading referral table to database",
    task_details="",
)
try:
    task_mgr.check_errors()

    referrals_df["Update Time HST"] = now_hst_str
    referrals_df["Update Time CST"] = now_str_formatted
    referrals_df["Update Time UTC"] = now_utc_str

    conn = PicHorneDashesConnection()

    conn.insert_with_progress(
        df=referrals_df,
        schema="dbo",
        tbl_nm="partners_in_care_referral_activity",
        dtypes={},
    )

    # updating the task tracker
    task["Details"] = "Successfully updated referral data in database"
    task["Success"] = True
    logger.info("Updated referral data in Database")

except Exception as e:
    task["Details"] = str(e)
    logger.error(
        f"Uploading new referral data to database failed with the following error:{e}"
    )
    print(e)

task_mgr.add_task(task)

This cell sends out an alert email if any of the above cells failed

In [10]:
if task_mgr._process_success != "Success":
    subject, body = task_mgr.generate_task_email(today_str)
    email_client = DsEmail()
    result = email_client.send_task_alert_email(subject, body, "partners_in_care")

Close out the logger

In [ ]:
logger_manager.cleanup()